### Building Chatbot With Multiple Tools Using Langgraph

#### Aim
Create a chatbot with tool capabilities from arxiv, wikipedia search and some functions

In [14]:
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper

In [15]:
api_wrapper_arxiv=ArxivAPIWrapper(top_k_results=2,doc_content_chars_max=500)
arxiv=ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
print(arxiv.name)

arxiv


In [16]:
arxiv.invoke("Attention iss all you need")

'Published: 2021-05-06\nTitle: Do You Even Need Attention? A Stack of Feed-Forward Layers Does Surprisingly Well on ImageNet\nAuthors: Luke Melas-Kyriazi\nSummary: The strong performance of vision transformers on image classification and other vision tasks is often attributed to the design of their multi-head attention layers. However, the extent to which attention is responsible for this strong performance remains unclear. In this short report, we ask: is the attention layer even necessary? Specifi'

In [17]:
api_wrapper_wiki=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=500)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name

'wikipedia'

In [18]:
wiki.invoke("What is machine learning")

'Page: Machine learning\nSummary: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without being explicitly programmed. Advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.\nStatistics and mathematical optimisation me'

In [19]:
from dotenv import load_dotenv
load_dotenv()

import os

os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [20]:
### Tavily Search Tool
from langchain_community.tools.tavily_search import TavilySearchResults

tavily = TavilySearchResults()

tavily.invoke("Provide me the recent AI news for 6th May 2026")

[{'title': 'State of AI: May 2026',
  'url': 'https://press.airstreet.com/p/state-of-ai-may-2026',
  'content': 'Coding, agents, and enterprise AI. Cognition was reported in talks for a follow-on at $25B, more than doubling the September 2025 mark of $10.2B. Cursor was reported in talks to raise $2B+ at $50B+ as enterprise revenue surged toward a $6B run-rate exit. Avoca hit unicorn status with $125M across three rounds at $1B for HVAC, plumbing, and roofing service agents (Series B led by Meritech and General Catalyst, Series A by Kleiner Perkins).\n\nDefense. Saronic raised $1.75B at $9.25B for autonomous naval vessels under the DoD’s Replicator initiative, more than doubling its mark from a year earlier.\n\nHealthcare AI. Qualified Health raised $125M for generative AI inside health-system clinical and operational workflows. [...] OpenAI’s GPT-5.5 followed just three weeks later with a near-identical capability profile: 2 of 10 end-to-end solves and 71.4% on expert tasks, carrying t

In [21]:
### Combine all the tools in the list

tools=[arxiv,wiki,tavily]

In [22]:
## Initialize my LLM model

from langchain_groq import ChatGroq

llm=ChatGroq(model="openai/gpt-oss-20b")

llm_with_tools=llm.bind_tools(tools)

In [23]:
from pprint import pprint
from langchain_core.messages import AIMessage, HumanMessage
llm_with_tools.invoke([HumanMessage(content=f"What is the recent AI News")])

AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "What is the recent AI News". They want recent AI news. We should provide up-to-date AI news. We might need to browse the web. Use tavily_search_results_json to get recent AI news. Let\'s search.', 'tool_calls': [{'id': 'fc_405a9542-127c-4c26-970e-6d68e6269938', 'function': {'arguments': '{"query":"recent AI news 2026"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 83, 'prompt_tokens': 291, 'total_tokens': 374, 'completion_time': 0.084705288, 'completion_tokens_details': {'reasoning_tokens': 51}, 'prompt_time': 0.015449353, 'prompt_tokens_details': None, 'queue_time': 0.154583856, 'total_time': 0.100154641}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e23fc997ca', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dfbce-f143-7313-95e2-120cfb9c7920-0',